In [2]:
import numpy as np
import pandas as pd
from itertools import product

def concentration_score(file_R0='R0.csv', 
                        file_sigma='sigma.csv',
                        true_R0=None,
                        true_sigma=None,
                        selected_indices=None):

    data_R0 = pd.read_csv(file_R0, header=None).values.ravel()
    data_sigma = pd.read_csv(file_sigma, header=None).values.ravel()
    
    # print(f"Loaded {len(data_R0)} {len(data_sigma)} samples")
    selected_R0 = data_R0[selected_indices]
    selected_sigma = data_sigma[selected_indices]
    
    d_R0    = ((selected_R0 - true_R0)    ** 2).mean()
    d_sigma = ((selected_sigma - true_sigma) ** 2).mean()
    return d_R0 + d_sigma

def compute_distance(filepath: str,
    standard_point: tuple,
    weights: tuple = None,
    metric: str = "euclidean",  # "euclidean", "manhattan", "chebyshev", "minkowski", "cosine"
    p: float = 3,               # only used when metric="minkowski"
    ):
                     
     # 1. Load
    df = pd.read_csv(filepath, header=None, names=["A", "B", "C", "D"])

    # 2. Min-Max normalization
    col_min = df.min()
    col_max = df.max()
    df_norm = (df - col_min) / (col_max - col_min)

    # 3. Normalize the standard point on the same scale
    standard = np.array(standard_point)
    standard_norm = (standard - col_min.values) / (col_max.values - col_min.values)

    # 4. Resolve weights (normalize so they sum to 1)
    if weights is not None:
        w = np.array(weights, dtype=float)
        if len(w) != 4:
            raise ValueError("weights must have exactly 4 values (wA, wB, wC, wD).")
        if np.any(w < 0):
            raise ValueError("All weights must be non-negative.")
        w = w / w.sum()          # normalize to sum = 1
    else:
        w = np.array([0.25, 0.25, 0.25, 0.25])   # equal weights

    # 5. Weighted different distance functions: Euclidean, manhattan, chebyshev, minkowski, cosine
    diff = (df_norm[["A", "B", "C", "D"]] - standard_norm).values
    if metric == "euclidean":
        dist = np.sqrt((w * diff ** 2).sum(axis=1))

    elif metric == "manhattan":
        dist = (w * np.abs(diff)).sum(axis=1)

    elif metric == "chebyshev":
        dist = (w * np.abs(diff)).max(axis=1)

    elif metric == "minkowski":
        dist = ((w * np.abs(diff) ** p).sum(axis=1)) ** (1 / p)

    elif metric == "cosine":
        dot     = (w * df_norm[["A", "B", "C", "D"]].values * standard_norm).sum(axis=1)
        norm_x  = np.sqrt((w * df_norm[["A", "B", "C", "D"]].values ** 2).sum(axis=1))
        norm_x0 = np.sqrt((w * standard_norm ** 2).sum())
        dist    = 1 - dot / (norm_x * norm_x0)

    else:
        raise ValueError(f"Unknown metric '{metric}'. Choose from: euclidean, manhattan, chebyshev, minkowski, cosine.")

    df_norm["distance"] = dist

    return df_norm["distance"]

In [5]:
##### setting: R0=1.5, sigma=0.6

stats = ["avg_prev", "div_prev", "npmi", "div_all_isolates"]
true_R0, true_sigma = 1.5, 0.6   # ← your true values
percentile = 0.05
standard_point=(12.39130435, 9.91206396, -0.34299727, 4.91916859)

# Grid over weights
weight_values = [i/10 for i in range(1, 10)]  # 0.1, 0.2, ..., 0.9

# files
file_ss="../../experimental_data/from_260312/ss_2params_R01p5.csv"
file_R0='../../experimental_data/from_260312/R0_samps_2params_R01p5.csv'
file_sigma='../../experimental_data/from_260312/sigma_samps_2params_R01p5.csv'


    # 1. Euclidean disntance for different weight vectors


results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="euclidean")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("euclidean distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    # 2. Manhattan distance for different weight vectors

# Grid over weights
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="manhattan")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("manhattan distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #3. Chebyshev distance for different weight vectors

results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="chebyshev")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("chebyshev distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #4. Minkowski distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="minkowski")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("minkowski distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #5. Cosine distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="cosine")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("cosine distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]

euclidean distance: 
                   weights     score     n
71   (0.1, 0.1, 0.8, 0.9)  0.139180  8000
80   (0.1, 0.1, 0.9, 0.9)  0.139536  8000
79   (0.1, 0.1, 0.9, 0.8)  0.139563  8000
70   (0.1, 0.1, 0.8, 0.8)  0.140170  8000
66   (0.1, 0.1, 0.8, 0.4)  0.140247  8000
75   (0.1, 0.1, 0.9, 0.4)  0.140397  8000
69   (0.1, 0.1, 0.8, 0.7)  0.140546  8000
809  (0.2, 0.1, 0.9, 0.9)  0.140616  8000
58   (0.1, 0.1, 0.7, 0.5)  0.140739  8000
76   (0.1, 0.1, 0.9, 0.5)  0.140749  8000
77   (0.1, 0.1, 0.9, 0.6)  0.140758  8000
78   (0.1, 0.1, 0.9, 0.7)  0.140883  8000
60   (0.1, 0.1, 0.7, 0.7)  0.141350  8000
806  (0.2, 0.1, 0.9, 0.6)  0.141378  8000
57   (0.1, 0.1, 0.7, 0.4)  0.141417  8000
59   (0.1, 0.1, 0.7, 0.6)  0.141468  8000
62   (0.1, 0.1, 0.7, 0.9)  0.141510  8000
808  (0.2, 0.1, 0.9, 0.8)  0.141532  8000
807  (0.2, 0.1, 0.9, 0.7)  0.141591  8000
798  (0.2, 0.1, 0.8, 0.7)  0.141629  8000
manhattan distance: 
                    weights     score     n
72    (0.1, 0.1, 0.9, 0.1)  0.1

In [3]:
##### setting: R0=2.0, sigma=0.6

stats = ["avg_prev", "div_prev", "npmi", "div_all_isolates"]
true_R0, true_sigma = 2.0, 0.6   # ← your true values
percentile = 0.05
standard_point=(27.17391304, 17.6703454,  -0.39818817,  5.03316728)

# Grid over weights
weight_values = [i/10 for i in range(1, 10)]  # 0.1, 0.2, ..., 0.9

# files
file_ss="../../experimental_data/from_260312/ss_2params_R02p0.csv"
file_R0='../../experimental_data/from_260312/R0_samps_2params_R02p0.csv'
file_sigma='../../experimental_data/from_260312/sigma_samps_2params_R02p0.csv'


    # 1. Euclidean disntance for different weight vectors


results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="euclidean")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("euclidean distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    # 2. Manhattan distance for different weight vectors

# Grid over weights
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="manhattan")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("manhattan distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #3. Chebyshev distance for different weight vectors

results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="chebyshev")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("chebyshev distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #4. Minkowski distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="minkowski")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("minkowski distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #5. Cosine distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="cosine")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("cosine distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]

euclidean distance: 
                    weights     score     n
1215  (0.2, 0.7, 0.1, 0.1)  0.113607  8000
6402  (0.9, 0.8, 0.1, 0.4)  0.114179  8000
486   (0.1, 0.7, 0.1, 0.1)  0.114393  8000
1377  (0.2, 0.9, 0.1, 0.1)  0.114617  8000
6483  (0.9, 0.9, 0.1, 0.4)  0.114669  8000
405   (0.1, 0.6, 0.1, 0.1)  0.114791  8000
657   (0.1, 0.9, 0.2, 0.1)  0.115105  8000
444   (0.1, 0.6, 0.5, 0.4)  0.115124  8000
1296  (0.2, 0.8, 0.1, 0.1)  0.115551  8000
706   (0.1, 0.9, 0.7, 0.5)  0.115696  8000
567   (0.1, 0.8, 0.1, 0.1)  0.115729  8000
624   (0.1, 0.8, 0.7, 0.4)  0.115837  8000
679   (0.1, 0.9, 0.4, 0.5)  0.115858  8000
1419  (0.2, 0.9, 0.5, 0.7)  0.116200  8000
5754  (0.8, 0.9, 0.1, 0.4)  0.116236  8000
469   (0.1, 0.6, 0.8, 0.2)  0.116341  8000
1420  (0.2, 0.9, 0.5, 0.8)  0.116471  8000
5672  (0.8, 0.8, 0.1, 0.3)  0.116551  8000
697   (0.1, 0.9, 0.6, 0.5)  0.116639  8000
688   (0.1, 0.9, 0.5, 0.5)  0.116681  8000
manhattan distance: 
                    weights     score     n
2135  (0.3

In [4]:
##### setting: R0=2.5, sigma=0.6

stats = ["avg_prev", "div_prev", "npmi", "div_all_isolates"]
true_R0, true_sigma = 2.5, 0.6   # ← your true values
percentile = 0.05
standard_point=(41.56521739, 20.81264058, -0.46938105,  5.88017828)

# Grid over weights
weight_values = [i/10 for i in range(1, 10)]  # 0.1, 0.2, ..., 0.9

# files
file_ss="../../experimental_data/from_260312/ss_2params_R02p5.csv"
file_R0='../../experimental_data/from_260312/R0_samps_2params_R02p5.csv'
file_sigma='../../experimental_data/from_260312/sigma_samps_2params_R02p5.csv'


    # 1. Euclidean disntance for different weight vectors


results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="euclidean")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("euclidean distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    # 2. Manhattan distance for different weight vectors

# Grid over weights
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="manhattan")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("manhattan distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #3. Chebyshev distance for different weight vectors

results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="chebyshev")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("chebyshev distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #4. Minkowski distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="minkowski")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("minkowski distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #5. Cosine distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="cosine")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("cosine distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]

euclidean distance: 
                    weights     score     n
594   (0.1, 0.8, 0.4, 0.1)  0.274307  8000
667   (0.1, 0.9, 0.3, 0.2)  0.275533  8000
684   (0.1, 0.9, 0.5, 0.1)  0.276535  8000
675   (0.1, 0.9, 0.4, 0.1)  0.276726  8000
603   (0.1, 0.8, 0.5, 0.1)  0.276951  8000
702   (0.1, 0.9, 0.7, 0.1)  0.277313  8000
666   (0.1, 0.9, 0.3, 0.1)  0.277637  8000
1431  (0.2, 0.9, 0.7, 0.1)  0.277900  8000
513   (0.1, 0.7, 0.4, 0.1)  0.278475  8000
576   (0.1, 0.8, 0.2, 0.1)  0.279662  8000
577   (0.1, 0.8, 0.2, 0.2)  0.279822  8000
522   (0.1, 0.7, 0.5, 0.1)  0.280001  8000
567   (0.1, 0.8, 0.1, 0.1)  0.280218  8000
648   (0.1, 0.9, 0.1, 0.1)  0.280498  8000
504   (0.1, 0.7, 0.3, 0.1)  0.280510  8000
604   (0.1, 0.8, 0.5, 0.2)  0.280723  8000
693   (0.1, 0.9, 0.6, 0.1)  0.280725  8000
658   (0.1, 0.9, 0.2, 0.2)  0.280865  8000
676   (0.1, 0.9, 0.4, 0.2)  0.280882  8000
657   (0.1, 0.9, 0.2, 0.1)  0.281152  8000
manhattan distance: 
                    weights     score     n
603   (0.1

In [ ]:
##### setting: R0=3.0, sigma=0.6

stats = ["avg_prev", "div_prev", "npmi", "div_all_isolates"]
true_R0, true_sigma = 3.0, 0.6   # ← your true values
percentile = 0.05
standard_point=()

# Grid over weights
weight_values = [i/10 for i in range(1, 10)]  # 0.1, 0.2, ..., 0.9

# files
file_ss="../../experimental_data/from_260312/ss_2params_R03p0.csv"
file_R0='../../experimental_data/from_260312/R0_samps_2params_R03p0.csv'
file_sigma='../../experimental_data/from_260312/sigma_samps_2params_R03p0.csv'


    # 1. Euclidean disntance for different weight vectors


results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="euclidean")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("euclidean distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    # 2. Manhattan distance for different weight vectors

# Grid over weights
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="manhattan")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("manhattan distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #3. Chebyshev distance for different weight vectors

results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="chebyshev")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("chebyshev distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #4. Minkowski distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="minkowski")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("minkowski distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #5. Cosine distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="cosine")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("cosine distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]

In [ ]:
##### setting: R0=3.5, sigma=0.6

stats = ["avg_prev", "div_prev", "npmi", "div_all_isolates"]
true_R0, true_sigma = 3.5, 0.6   # ← your true values
percentile = 0.05
standard_point=()

# Grid over weights
weight_values = [i/10 for i in range(1, 10)]  # 0.1, 0.2, ..., 0.9

# files
file_ss="../../experimental_data/from_260312/ss_2params_R03p5.csv"
file_R0='../../experimental_data/from_260312/R0_samps_2params_R03p5.csv'
file_sigma='../../experimental_data/from_260312/sigma_samps_2params_R03p5.csv'


    # 1. Euclidean disntance for different weight vectors


results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="euclidean")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("euclidean distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    # 2. Manhattan distance for different weight vectors

# Grid over weights
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="manhattan")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("manhattan distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #3. Chebyshev distance for different weight vectors

results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="chebyshev")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("chebyshev distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #4. Minkowski distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="minkowski")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("minkowski distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #5. Cosine distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="cosine")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("cosine distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]

In [5]:
##### setting: R0=4.0, sigma=0.6

stats = ["avg_prev", "div_prev", "npmi", "div_all_isolates"]
true_R0, true_sigma = 4.0, 0.6   # ← your true values
percentile = 0.05
standard_point=(136.2173913, 64.07875589,  -0.38683102,  14.22282713)

# Grid over weights
weight_values = [i/10 for i in range(1, 10)]  # 0.1, 0.2, ..., 0.9

# files
file_ss="../../experimental_data/from_260312/ss_2params_R04p0.csv"
file_R0='../../experimental_data/from_260312/R0_samps_2params_R04p0.csv'
file_sigma='../../experimental_data/from_260312/sigma_samps_2params_R04p0.csv'


    # 1. Euclidean disntance for different weight vectors


results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="euclidean")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("euclidean distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    # 2. Manhattan distance for different weight vectors

# Grid over weights
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="manhattan")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("manhattan distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #3. Chebyshev distance for different weight vectors

results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="chebyshev")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("chebyshev distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #4. Minkowski distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="minkowski")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("minkowski distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #5. Cosine distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="cosine")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("cosine distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]

euclidean distance: 
                    weights     score     n
5847  (0.9, 0.1, 0.2, 0.7)  0.431167  5000
5848  (0.9, 0.1, 0.2, 0.8)  0.431213  5000
4390  (0.7, 0.1, 0.2, 0.8)  0.431821  5000
4389  (0.7, 0.1, 0.2, 0.7)  0.432105  5000
4472  (0.7, 0.2, 0.2, 0.9)  0.432271  5000
5858  (0.9, 0.1, 0.3, 0.9)  0.432871  5000
2190  (0.4, 0.1, 0.1, 0.4)  0.433627  5000
5200  (0.8, 0.2, 0.2, 0.8)  0.433627  5000
5929  (0.9, 0.2, 0.2, 0.8)  0.433940  5000
5119  (0.8, 0.1, 0.2, 0.8)  0.434185  5000
5126  (0.8, 0.1, 0.3, 0.6)  0.434608  5000
5118  (0.8, 0.1, 0.2, 0.7)  0.434796  5000
5846  (0.9, 0.1, 0.2, 0.6)  0.435428  5000
5855  (0.9, 0.1, 0.3, 0.6)  0.435451  5000
4387  (0.7, 0.1, 0.2, 0.5)  0.435514  5000
5210  (0.8, 0.2, 0.3, 0.9)  0.435599  5000
5836  (0.9, 0.1, 0.1, 0.5)  0.435727  5000
5120  (0.8, 0.1, 0.2, 0.9)  0.435788  5000
3652  (0.6, 0.1, 0.1, 0.8)  0.435793  5000
4552  (0.7, 0.3, 0.2, 0.8)  0.435797  5000
manhattan distance: 
                  weights     score     n
80  (0.1, 0.

In [ ]:
##### setting: R0=4.5, sigma=0.6

stats = ["avg_prev", "div_prev", "npmi", "div_all_isolates"]
true_R0, true_sigma = 4.5, 0.6   # ← your true values
percentile = 0.05
standard_point=()

# Grid over weights
weight_values = [i/10 for i in range(1, 10)]  # 0.1, 0.2, ..., 0.9

# files
file_ss="../../experimental_data/from_260312/ss_2params_R04p5.csv"
file_R0='../../experimental_data/from_260312/R0_samps_2params_R04p5.csv'
file_sigma='../../experimental_data/from_260312/sigma_samps_2params_R04p5.csv'


    # 1. Euclidean disntance for different weight vectors


results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="euclidean")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("euclidean distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    # 2. Manhattan distance for different weight vectors

# Grid over weights
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="manhattan")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("manhattan distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #3. Chebyshev distance for different weight vectors

results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="chebyshev")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("chebyshev distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #4. Minkowski distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="minkowski")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("minkowski distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #5. Cosine distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="cosine")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("cosine distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]

In [ ]:
##### setting: R0=5.0, sigma=0.6

stats = ["avg_prev", "div_prev", "npmi", "div_all_isolates"]
true_R0, true_sigma = 5.0, 0.6   # ← your true values
percentile = 0.05
standard_point=()

# Grid over weights
weight_values = [i/10 for i in range(1, 10)]  # 0.1, 0.2, ..., 0.9

# files
file_ss="../../experimental_data/from_260312/ss_2params_R05p0.csv"
file_R0='../../experimental_data/from_260312/R0_samps_2params_R05p0.csv'
file_sigma='../../experimental_data/from_260312/sigma_samps_2params_R05p0.csv'


    # 1. Euclidean disntance for different weight vectors


results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="euclidean")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("euclidean distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    # 2. Manhattan distance for different weight vectors

# Grid over weights
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="manhattan")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("manhattan distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #3. Chebyshev distance for different weight vectors

results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="chebyshev")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("chebyshev distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #4. Minkowski distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="minkowski")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("minkowski distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #5. Cosine distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="cosine")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("cosine distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]